# nb03 Parity: every dashboard figure verified in DuckDB SQL and pandas

**Purpose**
- Recompute every number the published dashboard displays, in two independent engines, and compare them to each other and to fixed control values captured live from the DMHC (Department of Managed Health Care) report during diagnostics.
- The union of the 8 clean CSVs is rebuilt in both engines exactly the way Tableau's wildcard union builds it.
- Coverage: union integrity, control totals, the loss quarter lists, Medical Loss Ratio statistics and reconciliation, the per member per month math, and the Tangible Net Equity cushion.

**Prerequisites**
- Python 3 with `pandas` and `duckdb`. Run from the `tableau_plan_financials/notebooks/` folder, after nb02.

**Outputs**
- A pass or fail line per check and a final tally, mirrored to `nb03_sql_pandas_parity_cell_output.txt`.

In [1]:
# Step 0: mirror all printed output to a text file for easy sharing
import sys
from pathlib import Path

SINK_PATH = Path.cwd() / "nb03_sql_pandas_parity_cell_output.txt"

_orig_out = getattr(sys, "_nb_orig_stdout", sys.stdout)
_orig_err = getattr(sys, "_nb_orig_stderr", sys.stderr)
sys._nb_orig_stdout, sys._nb_orig_stderr = _orig_out, _orig_err

class _Tee:
    def __init__(self, stream, fh):
        self.stream, self.fh = stream, fh
    def write(self, data):
        self.stream.write(data)
        self.fh.write(data)
        self.fh.flush()
    def flush(self):
        self.stream.flush()
        self.fh.flush()

_sink = open(SINK_PATH, "w")
sys.stdout = _Tee(_orig_out, _sink)
sys.stderr = _Tee(_orig_err, _sink)
print(f"Mirroring cell output to {SINK_PATH.name} (attach this file in the chat)")

Mirroring cell output to nb03_sql_pandas_parity_cell_output.txt (attach this file in the chat)
files unioned: 8; duckdb rows: 162; pandas rows: 162
[PASS] total rows in the union: sql=162, pandas=162, expected 162
[PASS] distinct plans: sql=2, pandas=2, expected 2
[PASS] L.A. Care: rows: sql=81, pandas=81, expected 81
[PASS] L.A. Care: quarterly rows: sql=65, pandas=65, expected 65
[PASS] L.A. Care: annual rows: sql=16, pandas=16, expected 16
[PASS] L.A. Care: distinct quarterly periods equal quarterly rows: sql=65, pandas=65, expected 65
[PASS] Health Net Community Solutions: rows: sql=81, pandas=81, expected 81
[PASS] Health Net Community Solutions: quarterly rows: sql=65, pandas=65, expected 65
[PASS] Health Net Community Solutions: annual rows: sql=16, pandas=16, expected 16
[PASS] Health Net Community Solutions: distinct quarterly periods equal quarterly rows: sql=65, pandas=65, expected 65
[PASS] L.A. Care 2025 annual Total Revenue: sql=15800808336.0, pandas=15800808336.0, expect

In [2]:
# Step 1: build the union in BOTH engines, exactly as Tableau's wildcard union does
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
CLEAN_GLOB = str(PROJECT_ROOT / "data" / "clean" / "financial_summary_*.csv")

# engine 1: DuckDB reads and unions the files itself
con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE VIEW pf AS
    SELECT * FROM read_csv_auto('{CLEAN_GLOB}', union_by_name=true)
""")

# engine 2: pandas reads and unions the files itself
import glob
files = sorted(glob.glob(CLEAN_GLOB))
pf = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
pf["Period End Date"] = pd.to_datetime(pf["Period End Date"])

# the check harness
PASS = 0
FAIL = 0
def check(name, sql_value, pd_value, expected=None, tol=0.0):
    """One parity check: SQL vs pandas, and optionally vs a fixed expected value."""
    global PASS, FAIL
    def close(a, b):
        if a is None or b is None:
            return a is b
        if isinstance(a, float) or isinstance(b, float):
            return abs(float(a) - float(b)) <= tol
        return a == b
    ok = close(sql_value, pd_value) and (expected is None or close(sql_value, expected))
    PASS, FAIL = PASS + ok, FAIL + (not ok)
    tag = "PASS" if ok else "FAIL"
    exp = f", expected {expected}" if expected is not None else ""
    print(f"[{tag}] {name}: sql={sql_value}, pandas={pd_value}{exp}")

def sql1(q):
    """Run a query returning a single value."""
    return con.execute(q).fetchone()[0]

print(f"files unioned: {len(files)}; duckdb rows: {sql1('SELECT COUNT(*) FROM pf')}; pandas rows: {len(pf)}")

In [3]:
# Step 2: union integrity, the numbers behind the data source itself
check("total rows in the union", sql1("SELECT COUNT(*) FROM pf"), len(pf), expected=162)
check("distinct plans", sql1('SELECT COUNT(DISTINCT "Plan") FROM pf'), pf["Plan"].nunique(), expected=2)

for plan in ["L.A. Care", "Health Net Community Solutions"]:
    g = pf[pf["Plan"] == plan]
    check(f"{plan}: rows", sql1(f"""SELECT COUNT(*) FROM pf WHERE "Plan" = '{plan}'"""), len(g), expected=81)
    check(f"{plan}: quarterly rows",
          sql1(f"""SELECT COUNT(*) FROM pf WHERE "Plan" = '{plan}' AND "Period Type" = 'Quarterly'"""),
          len(g[g["Period Type"] == "Quarterly"]), expected=65)
    check(f"{plan}: annual rows",
          sql1(f"""SELECT COUNT(*) FROM pf WHERE "Plan" = '{plan}' AND "Period Type" = 'Annual'"""),
          len(g[g["Period Type"] == "Annual"]), expected=16)
    check(f"{plan}: distinct quarterly periods equal quarterly rows",
          sql1(f"""SELECT COUNT(DISTINCT "Period End Date") FROM pf
                   WHERE "Plan" = '{plan}' AND "Period Type" = 'Quarterly'"""),
          g[g["Period Type"] == "Quarterly"]["Period End Date"].nunique(), expected=65)

In [4]:
# Step 3: control totals captured LIVE from the DMHC report during diagnostics
# these anchor the whole pipeline to the source of truth
CONTROLS = [
    # (name, plan, period type, period end date, column, expected value)
    ("L.A. Care 2025 annual Total Revenue", "L.A. Care", "Annual", "2025-12-31", "Total Revenue", 15800808336),
    ("L.A. Care 2026 Q1 Total Revenue", "L.A. Care", "Quarterly", "2026-03-31", "Total Revenue", 3235767623),
    ("L.A. Care 2025 Q4 Total Revenue", "L.A. Care", "Quarterly", "2025-12-31", "Total Revenue", 3188919090),
    ("HNCS 2026 Q1 Total Revenue", "Health Net Community Solutions", "Quarterly", "2026-03-31", "Total Revenue", 3610424847),
    ("HNCS 2026 Q1 Net Income", "Health Net Community Solutions", "Quarterly", "2026-03-31", "Net Income-Loss", 110999890),
    ("HNCS 2025 Q4 Net Income (the loss)", "Health Net Community Solutions", "Quarterly", "2025-12-31", "Net Income-Loss", -18854385),
    ("HNCS 2025 Q3 Net Income", "Health Net Community Solutions", "Quarterly", "2025-09-30", "Net Income-Loss", 83928840),
    ("HNCS 2026 Q1 Medi-Cal enrollment", "Health Net Community Solutions", "Quarterly", "2026-03-31", "Medi-Cal Managed Care", 1494680),
]
for name, plan, ptype, date, col, exp in CONTROLS:
    s = sql1(f"""SELECT "{col}" FROM pf WHERE "Plan" = '{plan}'
                 AND "Period Type" = '{ptype}' AND "Period End Date" = DATE '{date}'""")
    p = pf[(pf["Plan"] == plan) & (pf["Period Type"] == ptype)
           & (pf["Period End Date"] == pd.Timestamp(date))][col].iloc[0]
    check(name, float(s), float(p), expected=float(exp), tol=0.01)

# the two ratio control values, reported to 2 decimals by the DMHC application
for name, date, exp in [("HNCS 2026 Q1 Health Expense Ratio", "2026-03-31", 91.62),
                        ("HNCS 2025 Q4 Health Expense Ratio", "2025-12-31", 93.71)]:
    s = sql1(f"""SELECT "Health Expense Ratio" FROM pf
                 WHERE "Plan" = 'Health Net Community Solutions'
                 AND "Period Type" = 'Quarterly' AND "Period End Date" = DATE '{date}'""")
    p = pf[(pf["Plan"] == "Health Net Community Solutions") & (pf["Period Type"] == "Quarterly")
           & (pf["Period End Date"] == pd.Timestamp(date))]["Health Expense Ratio"].iloc[0]
    check(name, round(float(s), 2), round(float(p), 2), expected=exp, tol=0.005)

In [5]:
# Step 4: the Quarterly Profit and Loss chart, loss quarters per plan
EXPECTED_LOSS = {
    "L.A. Care": ["2011 Q1", "2011 Q4", "2012 Q1", "2012 Q2", "2012 Q4", "2013 Q1", "2013 Q4",
                  "2014 Q3", "2016 Q3", "2018 Q2", "2019 Q3", "2020 Q2", "2020 Q3", "2020 Q4",
                  "2022 Q1", "2022 Q3", "2025 Q4"],
    "Health Net Community Solutions": ["2012 Q1", "2020 Q4", "2025 Q4"],
}
for plan, expected_list in EXPECTED_LOSS.items():
    s_list = [r[0] for r in con.execute(f"""
        SELECT "Quarter Label" FROM pf
        WHERE "Plan" = '{plan}' AND "Period Type" = 'Quarterly' AND "Net Income-Loss" < 0
        ORDER BY "Period End Date" """).fetchall()]
    g = pf[(pf["Plan"] == plan) & (pf["Period Type"] == "Quarterly") & (pf["Net Income-Loss"] < 0)]
    p_list = g.sort_values("Period End Date")["Quarter Label"].tolist()
    check(f"{plan}: loss quarter count", len(s_list), len(p_list), expected=len(expected_list))
    check(f"{plan}: loss quarter list matches", s_list == expected_list, p_list == expected_list, expected=True)

# the shared crisis quarter
s = sql1("""SELECT COUNT(DISTINCT "Plan") FROM pf
            WHERE "Period Type" = 'Quarterly' AND "Quarter Label" = '2025 Q4' AND "Net Income-Loss" < 0""")
p = pf[(pf["Period Type"] == "Quarterly") & (pf["Quarter Label"] == "2025 Q4")
       & (pf["Net Income-Loss"] < 0)]["Plan"].nunique()
check("2025 Q4 was a loss for BOTH plans", s, p, expected=2)

# cumulative 16 year net income per plan, the bottom line of the whole story
for plan in EXPECTED_LOSS:
    s = sql1(f"""SELECT SUM("Net Income-Loss") FROM pf
                 WHERE "Plan" = '{plan}' AND "Period Type" = 'Annual'""")
    p = pf[(pf["Plan"] == plan) & (pf["Period Type"] == "Annual")]["Net Income-Loss"].sum()
    check(f"{plan}: 2010 to 2025 cumulative annual net income (engines agree)", float(s), float(p), tol=0.01)
    print(f"    -> {plan} cumulative 2010 to 2025: ${float(s):,.0f}")

In [6]:
# Step 5: the Medical Loss Ratio chart, statistics and reconciliation
EXPECTED_RATIO = {
    # plan: (min, median, max) of the quarterly Health Expense Ratio, from nb02
    "Health Net Community Solutions": (74.90, 86.62, 106.21),
    "L.A. Care": (86.32, 94.36, 100.44),
}
for plan, (emin, emed, emax) in EXPECTED_RATIO.items():
    s_min, s_med, s_max = con.execute(f"""
        SELECT MIN("Health Expense Ratio"), MEDIAN("Health Expense Ratio"), MAX("Health Expense Ratio")
        FROM pf WHERE "Plan" = '{plan}' AND "Period Type" = 'Quarterly'""").fetchone()
    g = pf[(pf["Plan"] == plan) & (pf["Period Type"] == "Quarterly")]["Health Expense Ratio"]
    check(f"{plan}: ratio min", round(float(s_min), 2), round(float(g.min()), 2), expected=emin, tol=0.005)
    check(f"{plan}: ratio median", round(float(s_med), 2), round(float(g.median()), 2), expected=emed, tol=0.005)
    check(f"{plan}: ratio max", round(float(s_max), 2), round(float(g.max()), 2), expected=emax, tol=0.005)
    # quarters spent below the 85 floor (engines agree; value printed for the blog)
    s_below = sql1(f"""SELECT COUNT(*) FROM pf WHERE "Plan" = '{plan}'
                       AND "Period Type" = 'Quarterly' AND "Health Expense Ratio" < 85""")
    p_below = int((g < 85).sum())
    check(f"{plan}: quarters below the 85 floor (engines agree)", s_below, p_below)
    print(f"    -> {plan}: {s_below} of 65 quarters below 85")

# reconciliation: does the reported ratio equal 100 * medical expenses / revenue?
# informational, not asserted: the DMHC definition may use adjusted components
s_diff = sql1("""SELECT MAX(ABS("Health Expense Ratio" - 100.0 * "Total Medical Expenses" / "Total Revenue"))
                 FROM pf WHERE "Period Type" = 'Quarterly' AND "Total Revenue" > 0""")
q = pf[(pf["Period Type"] == "Quarterly") & (pf["Total Revenue"] > 0)]
p_diff = (q["Health Expense Ratio"] - 100.0 * q["Total Medical Expenses"] / q["Total Revenue"]).abs().max()
check("ratio reconciliation: engines agree on max deviation", round(float(s_diff), 2), round(float(p_diff), 2), tol=0.005)
print(f"    -> max |reported ratio - recomputed ratio| = {float(s_diff):.2f} points "
      f"({'definition matches simple division' if float(s_diff) < 0.5 else 'DMHC uses an adjusted definition; state this in the methods section'})")

In [7]:
# Step 6: the Per Member Per Month chart, recomputed from scratch
# member months = end of period Medi-Cal enrollment x 3 for a quarter (stated approximation)
LATEST = "2026-03-31"
for plan in ["L.A. Care", "Health Net Community Solutions"]:
    for col, label in [("Total Revenue", "Revenue PMPM"),
                       ("Total Medical Expenses", "Medical Cost PMPM"),
                       ("Total Administrative Expenses", "Admin Cost PMPM")]:
        s = sql1(f"""SELECT "{col}" / ("Medi-Cal Managed Care" * 3.0) FROM pf
                     WHERE "Plan" = '{plan}' AND "Period Type" = 'Quarterly'
                     AND "Period End Date" = DATE '{LATEST}'""")
        r = pf[(pf["Plan"] == plan) & (pf["Period Type"] == "Quarterly")
               & (pf["Period End Date"] == pd.Timestamp(LATEST))].iloc[0]
        p = r[col] / (r["Medi-Cal Managed Care"] * 3.0)
        check(f"{plan}: {label} 2026 Q1 (engines agree)", round(float(s), 2), round(float(p), 2), tol=0.005)
        print(f"    -> {plan} {label} 2026 Q1: ${float(s):,.2f}")
    # margin per member per month, the visible gap in the chart
    s_gap = sql1(f"""SELECT ("Total Revenue" - "Total Medical Expenses" - "Total Administrative Expenses")
                     / ("Medi-Cal Managed Care" * 3.0) FROM pf
                     WHERE "Plan" = '{plan}' AND "Period Type" = 'Quarterly'
                     AND "Period End Date" = DATE '{LATEST}'""")
    r = pf[(pf["Plan"] == plan) & (pf["Period Type"] == "Quarterly")
           & (pf["Period End Date"] == pd.Timestamp(LATEST))].iloc[0]
    p_gap = (r["Total Revenue"] - r["Total Medical Expenses"] - r["Total Administrative Expenses"]) / (r["Medi-Cal Managed Care"] * 3.0)
    check(f"{plan}: margin per member per month 2026 Q1 (engines agree)", round(float(s_gap), 2), round(float(p_gap), 2), tol=0.005)
    print(f"    -> {plan} margin per member per month 2026 Q1: ${float(s_gap):,.2f}")

In [8]:
# Step 7: the Tangible Net Equity chart, the cushion and the cliff
for plan in ["L.A. Care", "Health Net Community Solutions"]:
    # solvency: was actual TNE ever below the required minimum?
    s = sql1(f"""SELECT COUNT(*) FROM pf WHERE "Plan" = '{plan}'
                 AND "Period Type" = 'Quarterly' AND "TNE" < "Required TNE" """)
    g = pf[(pf["Plan"] == plan) & (pf["Period Type"] == "Quarterly")]
    p = int((g["TNE"] < g["Required TNE"]).sum())
    check(f"{plan}: quarters below the required TNE floor", s, p, expected=0)
    # smallest cushion in 16 years
    s_min = sql1(f"""SELECT MIN("TNE" - "Required TNE") FROM pf
                     WHERE "Plan" = '{plan}' AND "Period Type" = 'Quarterly'""")
    p_min = float((g["TNE"] - g["Required TNE"]).min())
    check(f"{plan}: smallest TNE cushion (engines agree)", round(float(s_min), 2), round(p_min, 2), tol=0.01)
    print(f"    -> {plan} smallest cushion: ${float(s_min):,.0f}")
    # latest quarter levels, as shown on the dashboard's right edge
    s_tne, s_req = con.execute(f"""SELECT "TNE", "Required TNE" FROM pf
        WHERE "Plan" = '{plan}' AND "Period Type" = 'Quarterly'
        AND "Period End Date" = DATE '2026-03-31'""").fetchone()
    r = g[g["Period End Date"] == pd.Timestamp("2026-03-31")].iloc[0]
    check(f"{plan}: 2026 Q1 TNE (engines agree)", float(s_tne), float(r["TNE"]), tol=0.01)
    check(f"{plan}: 2026 Q1 Required TNE (engines agree)", float(s_req), float(r["Required TNE"]), tol=0.01)
    print(f"    -> {plan} 2026 Q1: TNE ${float(s_tne):,.0f} vs required ${float(s_req):,.0f} "
          f"({100 * float(s_tne) / float(s_req):.0f}% of required)")

# the Health Net 2012 cliff: quantify the drop for the blog narrative
s_pre, = con.execute("""SELECT "TNE" FROM pf WHERE "Plan" = 'Health Net Community Solutions'
    AND "Period Type" = 'Quarterly' AND "Period End Date" = DATE '2012-09-30'""").fetchone()
s_post, = con.execute("""SELECT "TNE" FROM pf WHERE "Plan" = 'Health Net Community Solutions'
    AND "Period Type" = 'Quarterly' AND "Period End Date" = DATE '2012-12-31'""").fetchone()
hn = pf[(pf["Plan"] == "Health Net Community Solutions") & (pf["Period Type"] == "Quarterly")]
p_pre = float(hn[hn["Period End Date"] == pd.Timestamp("2012-09-30")]["TNE"].iloc[0])
p_post = float(hn[hn["Period End Date"] == pd.Timestamp("2012-12-31")]["TNE"].iloc[0])
check("HNCS TNE 2012 Q3 (engines agree)", float(s_pre), p_pre, tol=0.01)
check("HNCS TNE 2012 Q4 (engines agree)", float(s_post), p_post, tol=0.01)
print(f"    -> the 2012 cliff: ${float(s_pre):,.0f} (Q3) to ${float(s_post):,.0f} (Q4), "
      f"a ${float(s_pre) - float(s_post):,.0f} drop in one quarter")

In [9]:
# Step 8: the tally
print(f"\n{'=' * 50}")
print(f"PARITY RESULT: {PASS} passed, {FAIL} failed, {PASS + FAIL} total checks")
print(f"{'=' * 50}")

sys.stdout.flush()
print(f"\nAll printed output saved to: {SINK_PATH}")
print(f"File size: {SINK_PATH.stat().st_size:,} bytes")

**Next step**
- Attach `nb03_sql_pandas_parity_cell_output.txt` in the chat.
- Every figure quoted in the blog post comes from a PASS line or a printed `->` value in this output, nothing else.
- If any check fails, stop: the blog does not get written until the two engines and the dashboard agree.